## 03 — AI Search index
Creates the AI Search endpoint (once) and a Delta Sync index over the chunks table
with managed embeddings, then waits until the index is ready.

In [0]:
%pip install databricks-ai-search

In [0]:
dbutils.library.restartPython()

In [0]:
for key in ["catalog", "silver_schema", "gold_schema", "rag_schema"]:
    dbutils.widgets.text(key, "")

In [0]:
%run ./00_config

In [0]:
from databricks.ai_search.client import AISearchClient

client = AISearchClient()
ai_cfg = config["ai_search"]
endpoint_name = ai_cfg["endpoint_name"]

if client.endpoint_exists(endpoint_name):
    logger.info("AI Search endpoint %s already exists", endpoint_name)
else:
    logger.info("Creating AI Search endpoint %s", endpoint_name)
    client.create_endpoint_and_wait(name=endpoint_name, endpoint_type="STANDARD")

client.wait_for_endpoint(endpoint_name)
logger.info("AI Search endpoint %s is online", endpoint_name)

In [0]:
if client.index_exists(endpoint_name=endpoint_name, index_name=index_name):
    index = client.get_index(endpoint_name=endpoint_name, index_name=index_name)
    state = index.describe().get("status", {}).get("detailed_state")

    if state == "ONLINE_NO_PENDING_UPDATE":
        logger.info("Index %s is idle, triggering sync", index_name)
        index.sync()
    else:
        logger.info("Index %s is busy (%s), not triggering a new sync", index_name, state)
else:
    logger.info("Creating Delta Sync index %s from %s", index_name, chunks_table)
    index = client.create_delta_sync_index(
        endpoint_name=endpoint_name,
        index_name=index_name,
        source_table_name=chunks_table,
        primary_key="chunk_id",
        pipeline_type=ai_cfg["pipeline_type"],
        embedding_vector_column=ai_cfg["embedding_column"],
        embedding_dimension=ai_cfg["embedding_dimension"],
    )

In [0]:
index.wait_until_ready()

status = index.describe().get("status", {})
logger.info(
    "Index %s ready: state=%s, indexed rows=%s",
    index_name, status.get("detailed_state"), status.get("indexed_row_count"),
)